# E0: is a dense, warped, related task worth anything to a Bayes learner predicting a scarce one?

## Description

**Question.** A Meta-PFN would predict a scarce target task A (n_A = 0–8 points) from A's points *and* a dense source task B: a previous HPO run on a related dataset, whose landscape is related to A's through an unknown, non-uniform warp of the inputs. Before building one, the information has to be there. **Under a pair prior of this kind, how much better does a Bayes learner that knows the prior, but not the pair's warp or relatedness, predict A's held-out points than A alone does? And how much of the known-warp headroom survives having to infer the warp?** If a Bayes learner gains nothing, no amortized model can (the predecessor's E1 question, never run at full size).

**Setup.**
- **Pairs** (`pairs.py`). One latent function f is drawn from pfn-repro's BNN prior: the prior the frozen PFNs were trained on, on the same ECDF scale. B observes f directly (n_B = 32 uniform points). A observes it through a per-axis forward map T from A's box into a sub-window of B's (n_A ≤ 8 nested points, 32 held-out queries), all with the PFN's training noise. The strata are:
  - `identity`: T = id;
  - `window`: per axis T(x) = a + b x with b ∈ [0.4, 1];
  - `warp`: a window composed with a Kumaraswamy CDF, log α, log β ~ N(0, 0.5²);
  - `unrelated`: B observes an independent function.

  **Relatedness σ** perturbs A's BNN weights variance-preservingly. σ = 0 is the same function; σ = 0.05 and 0.1 give median Spearman(f_A, f_B) ≈ 0.9 and 0.7, measured at d = 1, 2 in the log (§2).
- **Predictors**, all from the frozen `pfn_large` (`oracle.py`):
  - `lower` = PFN(A), A alone;
  - `naive` = PFN(B ∪ A), pooling without registration;
  - `oracle_T` = PFN(B ∪ T(A)) at T(x\*), with the warp known;
  - `bayes`: the posterior predictive of a learner whose prior is the pair prior, a mixture over {unrelated, identity, 128–256 window and 128–256 warp hypotheses}. Each hypothesis is weighted by the PFN's chain-rule likelihood of A's points given B. It treats B as exact data under each hypothesis, so at σ > 0 it is a model-averaging learner rather than the exact Bayes one: a *lower* bound on what B is worth.
- **Grid.** d ∈ {1, 2} × {identity, window, warp} × σ ∈ {0, 0.05, 0.1}, plus `unrelated`, for 20 cells. Pilot: 50 pairs per cell (pair split 0). Confirm: fresh pairs (split 1).

**What to look for.**
- **Gains over A alone**, in nats per held-out query, with 95% pair-bootstrap CIs, as a function of n_A:
  - `gain_bayes` (what B is worth);
  - `gain_oracle` (with T known);
  - `gain_naive` (without registration);
  - G = gain_bayes / gain_oracle (the share of the headroom that survives inferring T).
- **Diagnostics.**
  - the posterior mass per relation kind;
  - the particles' ESS;
  - the oracle's Monte-Carlo error: an independent hypothesis set on 10 pairs per cell.

**Decision rule (fixed before the run; `analysis/headroom.py::verdict`).**
- **INVALID** unless, on identity pairs at σ = 0 and n_A = 8, the Bayes learner recovers ≥ 90% of the oracle gain, and the oracle gain is > 0 in every σ = 0 gate cell.
- **Validity.** A gate cell whose oracle MC error exceeds 0.025 nats is imprecise. An imprecise oracle under-estimates gains, so it can block KILL but never support it.
- **KILL** (stop the Meta-PFN line on this prior family) if in every gate cell (window and warp × σ ∈ {0, 0.05, 0.1} × n_A ∈ {2, 4, 8} × d ∈ {1, 2}) the CI upper bound of gain_bayes is < 0.05 nats.
- **GO** (build and train a Meta-PFN on this prior) if, in the go cells (window and warp at **σ = 0.1**, the least related level, × n_A ∈ {2, 4} × both d), the CI lower bound of gain_bayes is > 0.10 nats **and** G > 0.25.
- **NARROW** otherwise: restrict the prior to where the gain holds, then re-run.
- *Safety (reported with the verdict).* On unrelated pairs the CI lower bound of gain_bayes should stay above −0.05, so that negative transfer is avoided by the learner itself.

**Limits.**
- Per-axis warps only, no rotations or general axis coupling. That is the realistic HPO case (per-axis shifts and stretches), and it matches the predecessor's spec v4 at L1–L3.
- Noise is the BNN prior's own. No failures and no output maps per task.
- B's design is uniform, not trace-biased.
- d ≤ 2: the particle oracle degrades with the number of warp parameters, and the ESS and MC error say where.
- The chain-rule likelihood depends on the order of A's design. It is exact only for a Bayes-optimal PFN.


In [ ]:
from meta_pfn.analysis import load_snapshot, paired_diff, show_header, summarize

SNAPSHOT = "results/headroom/<sweep_id>"  # set after `just snapshot runs/headroom/<sweep_id>`
snap = load_snapshot(SNAPSHOT)
show_header(snap)  # how it was run + configuration, generated from the snapshot

## Results

In [ ]:
# Aggregates use DONE members only; n_failed / n_started / n_missing say what is not in them yet.
summarize(snap.runs, "<metric>")

## Verdict

The GO / NO-GO criterion from the description, evaluated, in one line. Record it with `just decide`.